# ISL Sign Recognition — Retrain Transformer for 81% Accuracy (263 classes)

**Target model:** `include_no_cnn_transformer_small.pth` — 81.34% accuracy

**Dataset:** Full INCLUDE dataset — 263 ISL sign classes

**Keypoints needed:** `keypoint/` folder (include_train/val/test_keypoints)

**Branch:** `cnn_new`

**Time:** ~45 min on Colab GPU

In [ ]:
# ── Step 1: Clone cnn_new branch ─────────────────────────────────────
!git clone --branch cnn_new --single-branch https://github.com/BishalDubey27/Major_Project.git
%cd Major_Project
!ls

In [ ]:
# ── Step 2: Install dependencies ─────────────────────────────────────
!pip install mediapipe==0.10.31 transformers==4.44.0 timm joblib tqdm scikit-learn -q
print('Done')

In [ ]:
# ── Step 3: Upload keypoint folder (full INCLUDE 263 classes) ─────────
# Zip your local 'keypoint' folder and upload here
# On Windows: right-click keypoint folder → Send to → Compressed (zipped) folder
# NOTE: This folder is large (~500MB). Google Drive option below is recommended.

from google.colab import files
import zipfile, os

uploaded = files.upload()  # Upload keypoint.zip

for fname in uploaded.keys():
    with zipfile.ZipFile(fname, 'r') as z:
        z.extractall('/content/')
    print('Extracted:', fname)

# Verify
for split in ['include_train_keypoints', 'include_val_keypoints', 'include_test_keypoints']:
    path = f'/content/keypoint/{split}'
    if os.path.exists(path):
        n = len([f for f in os.listdir(path) if f.endswith('.json')])
        print(f'{split}: {n} files')
    else:
        print(f'{split}: NOT FOUND - check zip structure')

In [ ]:
# ── Step 3 (Recommended): Load from Google Drive ──────────────────────
# Upload keypoint.zip to your Google Drive first, then use this:

# from google.colab import drive
# drive.mount('/content/drive')
# import zipfile
# with zipfile.ZipFile('/content/drive/MyDrive/keypoint.zip', 'r') as z:
#     z.extractall('/content/')
# print('Extracted from Drive')

In [ ]:
# ── Step 4: Train the model (263 classes, full INCLUDE) ───────────────
import os
os.chdir('/content/Major_Project/INCLUDE')

!python runner.py \
    --dataset include \
    --model transformer \
    --transformer_size small \
    --data_dir /content/keypoint \
    --save_path /content/ \
    --epochs 50 \
    --batch_size 128 \
    --learning_rate 1e-4 \
    --use_augs

In [ ]:
# ── Step 5: Verify the trained model ─────────────────────────────────
import torch, glob

pth_files = glob.glob('/content/*.pth')
print('Saved models:', pth_files)

if pth_files:
    cp = torch.load(pth_files[0], map_location='cpu', weights_only=False)
    print('Val accuracy:', round(cp.get('score', 0) * 100, 2), '%')
    print('Output classes:', cp['model']['l2.weight'].shape[0])  # should be 263
    print('Input size:', cp['model']['l1.weight'].shape[1])       # should be 134

In [ ]:
# ── Step 6: Test accuracy on test set ────────────────────────────────
import sys, json, numpy as np, pandas as pd
sys.path.insert(0, '/content/Major_Project')

from INCLUDE.models.transformer import Transformer
from INCLUDE.configs import TransformerConfig

with open('/content/Major_Project/INCLUDE/label_maps/label_map_include.json') as f:
    label_map = json.load(f)
idx_to_label = {v: k for k, v in label_map.items()}

cp = torch.load(pth_files[0], map_location='cpu', weights_only=False)
config = TransformerConfig(size='small')
model = Transformer(config=config, n_classes=263)
state = {k.replace('module.', ''): v for k, v in cp['model'].items()}
model.load_state_dict(state)
model.eval()

correct = 0; total = 0
test_dir = '/content/keypoint/include_test_keypoints'
for fname in sorted(os.listdir(test_dir)):
    if not fname.endswith('.json'): continue
    label_from_file = fname.split('_')[0]
    if label_from_file not in label_map: continue
    row = pd.read_json(os.path.join(test_dir, fname), typ='series')
    def combine_xy(x, y):
        x, y = np.array(x), np.array(y)
        _, l = x.shape
        return np.concatenate([x.reshape(-1,l,1), y.reshape(-1,l,1)], -1).astype(np.float32)
    def interp(arr):
        ax = pd.DataFrame(arr[:,:,0]).interpolate(method='linear', limit_direction='both').to_numpy() * 1920
        ay = pd.DataFrame(arr[:,:,1]).interpolate(method='linear', limit_direction='both').to_numpy() * 1080
        return np.stack([ax, ay], -1)
    pose = interp(combine_xy(row.pose_x, row.pose_y))
    h1   = interp(combine_xy(row.hand1_x, row.hand1_y))
    h2   = interp(combine_xy(row.hand2_x, row.hand2_y))
    data = np.concatenate([pose.reshape(-1,50), h1.reshape(-1,42), h2.reshape(-1,42)], -1)
    T = data.shape[0]
    data = data[:200] if T >= 200 else np.pad(data, ((0,200-T),(0,0)), 'constant')
    mean = data.mean(); std = data.std() + 1e-8
    data = np.nan_to_num((data-mean)/std, nan=0.0, posinf=0.0, neginf=0.0)
    tensor = torch.FloatTensor(data).unsqueeze(0)
    with torch.no_grad():
        pred = torch.softmax(model(tensor), dim=-1)[0].argmax().item()
    predicted = idx_to_label.get(pred, 'unknown')
    ok = row.label.lower() == predicted.lower()
    total += 1
    if ok: correct += 1

print(f'Test accuracy: {correct}/{total} = {round(100*correct/max(total,1),1)}%')

In [ ]:
# ── Step 7: Download the trained model ───────────────────────────────
from google.colab import files
files.download(pth_files[0])
print('Downloaded:', pth_files[0])
print()
print('Next steps:')
print('1. Place the downloaded .pth file in Major_Project/INCLUDE/ folder')
print('2. Rename it to: include_no_cnn_transformer_small.pth')
print('3. unified_app.py already uses this file with label_map_include.json (263 classes)')
print('4. Sign recognition will now work at ~81% accuracy!')